# Hyperparameter Tuning

SageMaker's managed `HyperparameterTuner` is control-plane machinery that
local mode does not emulate — but the *idea* it teaches (train candidate
configurations, compare a held-out metric) maps directly onto multiple
training jobs. Here we run a small sweep of `n_estimators`/`max_depth`
combinations as real local training jobs and read each job's `metrics.json`
sidecar out of its `model.tar.gz`.


## Environment & local SageMaker session

In [ ]:
import os

# Kernel env on PATH so %%bash cells can call feast/aws/docker directly; the
# aws CLI and Feast binaries live in the workspace venv (/opt/mlops-venv).
os.environ["PATH"] = "/opt/mlops-venv/bin:" + os.environ["PATH"]
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

# This project's own moto (see docker-compose.yml) — NOT the lab's shared one.
os.environ["MOTO_URL"] = "http://telco-customer-churn-moto:5000"

In [ ]:
import os
from pathlib import Path

import yaml
from sagemaker_local.config import LocalModeConfig
from sagemaker_local.patches import cleanup_stale_serving_containers
from sagemaker_local.session import make_local_session

# A killed run can leave a serving container bound to this project's host
# port; drop leftovers before we start (idempotent).
cleanup_stale_serving_containers()

# SAGEMAKER_LOCAL_REPO_PATH must resolve identically on the host and inside
# this container — local mode bind-mounts it into job/serving containers.
PROJECT_DIR = (
    Path(os.environ["SAGEMAKER_LOCAL_REPO_PATH"])
    / "projects"
    / "telco_customer_churn"
)

cfg = LocalModeConfig(
    **yaml.safe_load((PROJECT_DIR / "configs" / "local.yaml").read_text())
)
boto_session, sm_session = make_local_session(cfg)

TRAIN_IMAGE = "telco-customer-churn:train"
INFERENCE_IMAGE = "telco-customer-churn:inference"
print("s3 endpoint:", cfg.s3_endpoint_url, "| serving port:", cfg.serving_port)

## Rebuild the training channel

Same Feast retrieval as the training stage — each notebook stands alone.


In [ ]:
import pandas as pd
from feast import FeatureStore

fs = FeatureStore(repo_path="../feature_repo")
labels = pd.read_parquet("../data/processed/churn_labels.parquet")
training_df = fs.get_historical_features(
    entity_df=labels,
    features=[
        f"customer_subscription:{name}"
        for name in [
            "senior_citizen",
            "partner",
            "dependents",
            "tenure_months",
            "paperless_billing",
            "month_to_month",
            "one_year_contract",
            "electronic_check",
            "dsl",
            "fiber_optic",
            "streaming_tv",
            "streaming_movies",
            "num_add_ons",
            "monthly_charges",
            "total_charges",
        ]
    ],
).to_df()
training_df.to_csv("../data/processed/train.csv", index=False)
training_df.shape

In [ ]:
%%bash
aws s3 cp ../data/processed/train.csv     "s3://telco-customer-churn/input/train/train.csv"     --endpoint-url "$MOTO_URL"

## Run the sweep

Three jobs over `n_estimators`; `SM_HPS` carries each candidate into the
container as the job's hyperparameters.


In [ ]:
from sagemaker.estimator import Estimator


def fit_candidate(n_estimators: int, max_depth: int) -> str:
    est = Estimator(
        entry_point="train.py",
        source_dir=str(PROJECT_DIR / "src" / "telco_customer_churn"),
        image_uri=TRAIN_IMAGE,
        role=cfg.role_arn,
        instance_type="local",
        instance_count=1,
        sagemaker_session=sm_session,
        output_path=f"s3://{cfg.bucket}/models",
        hyperparameters={
            "n_estimators": n_estimators,
            "max_depth": max_depth,
        },
    )
    est.fit({"train": f"s3://{cfg.bucket}/input/train/"})
    return est.model_data


candidates = {(50, 3): None, (100, 4): None, (200, 6): None}
for n_est, depth in candidates:
    print(f"training n_estimators={n_est} max_depth={depth}")
    candidates[(n_est, depth)] = fit_candidate(n_est, depth)
candidates

## Collect the metrics

Each `model.tar.gz` carries the holdout `metrics.json` the job wrote —
download each artifact and line the candidates up.


In [ ]:
import json
import tarfile
import tempfile

import boto3

s3 = boto3.client("s3", endpoint_url=os.environ["MOTO_URL"])
results = []
for (n_est, depth), model_data in candidates.items():
    bucket, key = model_data.replace("s3://", "").split("/", 1)
    with tempfile.NamedTemporaryFile(suffix=".tar.gz") as tmp:
        s3.download_file(bucket, key, tmp.name)
        with tarfile.open(tmp.name) as tar:
            metrics = json.load(tar.extractfile("metrics.json"))
    results.append(
        {
            "n_estimators": n_est,
            "max_depth": depth,
            **metrics,
            "model_data": model_data,
        }
    )

results_df = pd.DataFrame(results).sort_values("auc", ascending=False)
results_df

In [ ]:
ax = results_df.plot(
    x="n_estimators",
    y=["accuracy", "auc"],
    kind="bar",
    rot=0,
    figsize=(8, 4),
    ylim=(0.5, 1.0),
)
ax.set_ylabel("holdout metric")
ax.set_title("Sweep results")
results_df.head(1)

The best candidate's `model_data` URI is what a pipeline would promote to
deployment — the tuning stage's output is a *choice*, not just a number.
